# Factoid Test Evidence-Support Check

Counts a factoid question as evidence-supported when at least one official gold alias occurs literally in its provided BioASQ snippets after the project normalization used for evidence-grounded SFT.


In [ ]:
import json
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / 'Task-Structured Counterfactual Preference Mining']
PROJECT_ROOT = next((path for path in candidates if (path / 'cse_dpo').is_dir()), None)
assert PROJECT_ROOT is not None, 'Run this notebook from the project or workspace directory.'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cse_dpo.normalize_set_answers import normalize_answer_surface

TEST_DIR = PROJECT_ROOT / 'data/Task13BTest'
TEST_FILES = sorted(TEST_DIR.glob('*_golden.json'))
assert len(TEST_FILES) == 4, f'Expected four golden test batches, found: {TEST_FILES}'
print('Test batches:', [path.name for path in TEST_FILES])


In [ ]:
def flatten_aliases(value):
    """BioASQ factoid exact_answer values are nested lists of alias strings."""
    if isinstance(value, str):
        return [value]
    if isinstance(value, list):
        return [alias for item in value for alias in flatten_aliases(item)]
    return []


def supported_aliases(question):
    snippets = question.get('snippets', [])
    evidence = ' '.join(
        snippet.get('text', '') for snippet in snippets if isinstance(snippet, dict)
    )
    normalized_evidence = normalize_answer_surface(evidence)
    return [
        alias
        for alias in flatten_aliases(question.get('exact_answer', []))
        if (normalized_alias := normalize_answer_surface(alias))
        and f' {normalized_alias} ' in f' {normalized_evidence} '
    ]


rows = []
for path in TEST_FILES:
    payload = json.loads(path.read_text(encoding='utf-8'))
    for question in payload['questions']:
        if question.get('type') != 'factoid':
            continue
        aliases = flatten_aliases(question.get('exact_answer', []))
        supported = supported_aliases(question)
        rows.append({
            'batch': path.stem,
            'question_id': question['id'],
            'question': question.get('body', ''),
            'gold_alias_count': len(aliases),
            'supported_aliases': supported,
            'is_evidence_supported': bool(supported),
        })

total = len(rows)
supported = sum(row['is_evidence_supported'] for row in rows)
print(f'Combined: {supported} / {total} evidence-supported factoid questions')
print(f'Evidence-only maximum MRR: {supported / total:.6f}')

for batch in sorted({row['batch'] for row in rows}):
    batch_rows = [row for row in rows if row['batch'] == batch]
    batch_supported = sum(row['is_evidence_supported'] for row in batch_rows)
    print(f'{batch}: {batch_supported} / {len(batch_rows)}')

unsupported = [row for row in rows if not row['is_evidence_supported']]
print(f'Unsupported under this literal-alias rule: {len(unsupported)}')
unsupported[:3]
